# PYSPARK E DATAFRAMES

### CRIAR TABELA queimadas_raw



In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.queimadas_raw
USING delta -- final será Delta, habilita replace table
AS
SELECT * FROM read_files(
    '/Volumes/workspace/default/my_teste_raw_data/queimadas_brasil_ficticio_raw.csv',
    format => 'csv',
    header => 'true',
    inferSchema => 'true',
    schemaEvolutionMode => 'none' -- desabilita _rescued_data col (schema inference)
    
);


In [0]:
%sql
SELECT * FROM workspace.default.queimadas_raw LIMIT 10 ;

Lendo tabela --> Criando um datafrane com pyspark

In [0]:
database_name = 'workspace.default'
table_name = 'queimadas_raw'

df_raw = spark.read.table(f'{database_name}.{table_name}')
display(df_raw.limit(10))



Limpeza de dados com pyspark


In [0]:
from pyspark.sql.functions import col
df_raw.filter(col("temperatura_c") <= 25).count()

In [0]:
# quantidade de linhas com área queimada maior que 200ha por localidade (long, lat)
display(
    df_raw.filter(col('area_queimada_ha') > 200)
    .groupBy('latitude', 'longitude', 'uf')
    .count()
    .orderBy('count', ascending=False)
    .limit(100)
)



CRIANDO UMA NOVA COLUNA PARA PK

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

# Cria um dataframe com uma coluna 'pk' que é um id incremental
database_name = 'workspace.default'
table_name = 'queimadas_raw'
df_raw = spark.read.table(f'{database_name}.{table_name}')
display(df_raw.limit(10))

df_pk = df_raw.withColumn('pk', monotonically_increasing_id())
display(df_pk.limit(100))

# Criaria uma nova tabela com a coluna 'pk'
#df_pk.write.mode('overwrite').saveAsTable('workspace.default.queimadas_raw_pk')

ESCREVENDO UMA TABELA COM PYSPARK

In [0]:
df_pk.write.mode('overwrite').saveAsTable('workspace.default.queimadas_raw_pk')
df_raw_pk = spark.read.table('workspace.default.queimadas_raw_pk')
display(df_raw_pk.limit(100))

CONVERTER DADOS DA TABELA PARA UM VALOR ESPECIFICO

In [0]:
#converter os dados floot de temperatura_c em dados int
df_raw_pk = df_raw_pk.withColumn('temperatura_c', df_raw_pk['temperatura_c'].cast('int'))
display(df_raw_pk.limit(100))


CRIAR COLUNA COM VALORES NEGATIVOS PARA TESTARMOS A ALTERAÇÃO DESSES DADOS COM REPLACE


In [0]:
from pyspark.sql.functions import lit

df_raw_pk = df_raw_pk.withColumn('dados_negativos', lit(-999))
display(df_raw_pk.limit(100))


AGORA VAMOS ALTERAR OS VALORES DA COLUNA DADOS_NEGATIVOS DO DATASET

In [0]:
df_raw_pk = df_raw_pk.replace(-999, None, 'dados_negativos')
display(df_raw_pk.limit(100))


CONTINUAREMOS OS ESTUDOS EM OUTRO NOTEBOOK